**Unidad de análisis**: La reservación. Cada fila corresponde a una reserva. No son huéspedes.

**Target**: `is_canceled` (1 = la reserva fue cancelada, 0 = no fue cancelada).

**Momento de la predicción**: momento en que se solicita la reserva (esto es lo que yo propongo). Entiendo que tendría un sentido operativo para el hotel estimar si la reseva será cancelada al momento en que se solicita la reserva para poder gestionar las plazas disponibles, o el personal necesario. 
    -> Cualquier variable que se conozca posterior al momento de la reserva no puede ser considerada al momento del análisis

**Utilidad potencial del modelo**: anticipar cancelaciones permite al hotel ajustar políticas de overbooking, de depósito/garantía, y de asignación de recursos.
 
Considerando lo anterior, debemos obtener la variable de “la fecha en que se realiza la reserva” (booking_date) que la podemos estimar como 

    booking_date ≈ arrival_date − lead_time
    
Creo que es la partición "natural" si pensamos en la problematica: "hoy tomo la reserva, hoy predigo".
Esto es una decisión discutible y *REVISAR EN GRUPO*.  


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

## Carga de datos

In [ ]:
DATA_PATH = r"C:\Users\evitar\OneDrive\MIA\PYTHON\TP\hotel_booking_demand\data\raw\hotel_bookings.csv" ### RUTA DE USTEDES

df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

* Eliminacion de duplicados

In [ ]:
df_clean = df_raw.drop_duplicates()
print(f"Filas sin duplicados: {df_clean.shape[0]}")

## Partición temporal (train / validation / test)

1. Construimos `booking_date` combinando las columnas de fecha de llegada (`arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month`) y restando `lead_time` días.

2. **Proporciones elegidas:** 70% train / 15% validation / 15% test, ordenado cronológicamente (no al azar), para que validation y test representen períodos *posteriores* a train — que es la situación real en producción.

Esta proporción es la convención conversada en clase, no una exigencia de la consigna. Según leí se puede modificar. Cuando modificariamos? cuando por ejemplo, si al ver la distribución temporal el 15% final resulta ser un período atípico, un solo mes... algo que nos haga cambiar de opinion...

In [ ]:
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9, "October": 10, "November": 11, "December": 12,
}

df_clean["arrival_date"] = pd.to_datetime(
    dict(
        year=df_clean["arrival_date_year"],
        month=df_clean["arrival_date_month"].map(month_map),
        day=df_clean["arrival_date_day_of_month"]
    )
)

df_clean["booking_date"] = df_clean["arrival_date"] - pd.to_timedelta(df_clean["lead_time"], unit="D")

df_clean = df_clean.sort_values("booking_date").reset_index(drop=True)

print(df_clean["booking_date"].min(), "->", df_clean["booking_date"].max())

In [ ]:
n = len(df_clean)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = df_clean.iloc[:train_end].copy()
df_val = df_clean.iloc[train_end:val_end].copy()
df_test = df_clean.iloc[val_end:].copy()

for name, part in [("train", df_train), ("validation", df_val), ("test", df_test)]:
    print(
        f"{name:11s} | filas: {len(part):6d} "
        f"| desde {part['booking_date'].min().date()} hasta {part['booking_date'].max().date()} "
        f"| % canceled {part['is_canceled'].mean() * 100:.2f}% "
    )

In [ ]:
df_raw.head()
df_raw.tail()

# Creamos DF para el EDA - test no se analiza en EDA

In [ ]:
df = pd.concat([df_train, df_val], ignore_index=True)
df.head()

In [ ]:
for name, part in [("train_eda", df), ("train", df_train), ("validation", df_val), ("test", df_test)]:
    print(
        f"{name:11s} | filas: {len(part):6d} "
        f"| desde {part['booking_date'].min().date()} hasta {part['booking_date'].max().date()} "
        f"| % canceled {part['is_canceled'].mean() * 100:.2f}% "
    )

In [ ]:
# Mostrar tabla completa
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

resumen = []

for col in df.columns:

    n_unicos = df[col].nunique(dropna=False)

    es_str = (
        pd.api.types.is_object_dtype(df[col]) or
        pd.api.types.is_string_dtype(df[col])
    )

    es_int_pocos_niveles = (
        pd.api.types.is_integer_dtype(df[col]) and
        n_unicos <= 3
    )

    if es_str or es_int_pocos_niveles:

        resumen.append({
            "variable": col,
            "tipo": str(df[col].dtype),
            "n_valores": n_unicos,
            "valores": list(df[col].drop_duplicates()),
            "n_faltantes": df[col].isna().sum(),
            "%_faltantes": round(df[col].isna().mean() * 100, 2)
        })

tabla_valores = pd.DataFrame(resumen)

display(tabla_valores)

In [ ]:
tabla_valores[tabla_valores["n_valores"] == 2]

* Identifico las variables segun su tipo para tratarlas de la misma manera

In [ ]:
print(f"df: {df.shape[0]:,} filas x {df.shape[1]} columnas\n")

# agent y company son identificadores, no magnitudes: los forzamos a string
for col in ["agent", "company"]:
    if col in df.columns:
        df[col] = df[col].astype("Int64").astype(str).replace("<NA>", np.nan)

df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"], errors="coerce")

# Variables dicotómicas como categóricas
for col in ["is_canceled", "is_repeated_guest"]:
    if col in df.columns:
        df[col] = df[col].astype("category")

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Numéricas ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"\nFechas ({len(df.select_dtypes(include=['datetime64[ns]']).columns)}): {df.select_dtypes(include=['datetime64[ns]']).columns.tolist()}")

In [ ]:
df[numeric_cols].describe().T

In [ ]:
df[categorical_cols].describe().T   

In [ ]:
tablas = []

for var in categorical_cols:

    freq = (
        df[var]
        .value_counts(dropna=False)
        .reset_index()
    )

    freq.columns = ["categoria", "frecuencia"]

    freq["variable"] = var

    tablas.append(freq)

tabla_freq = pd.concat(tablas, ignore_index=True)

tabla_freq = tabla_freq[
    ["variable", "categoria", "frecuencia"]
]
tabla_freq.head()

In [ ]:
cancelaciones = (
    df[df["is_canceled"] == 1]
    .groupby(pd.Grouper(key="arrival_date", freq="ME"))
    .size()
    .reset_index(name="n_cancelaciones")
)

# Gráfico
plt.figure(figsize=(10, 5))
plt.plot(
    cancelaciones["arrival_date"],
    cancelaciones["n_cancelaciones"],
    marker="o"
)

plt.title("Cancelaciones mensuales por fecha de llegada")
plt.xlabel("Mes")
plt.ylabel("Número de cancelaciones")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Filtrar cancelaciones
cancelaciones = (
    df[df["is_canceled"] == 1]
    .groupby(pd.Grouper(key="booking_date", freq="ME"))
    .size()
    .reset_index(name="n_cancelaciones")
)

# Gráfico
plt.figure(figsize=(10, 5))
plt.plot(
    cancelaciones["booking_date"],
    cancelaciones["n_cancelaciones"],
    marker="o"
)

plt.title("Cancelaciones mensuales por fecha en que se solicita la reserva")
plt.xlabel("Mes")
plt.ylabel("Número de cancelaciones")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

* Quito de las variabes categoricas las variables que solo podrian ser medidas despues del dia de reserva: 
1. assigned_room_type
2. reservation_status

In [ ]:
var_cat = ('hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'customer_type')
df["is_canceled"] = df["is_canceled"].astype(int)
for var in var_cat:

    resumen = (
    df
    .groupby(var, dropna=False)
    .agg(
        n=("is_canceled", "size"),
        cancelaciones=("is_canceled", "sum"),
        prop_cancelados=("is_canceled", "mean")
    )
).

    resumen["prop_cancelados"] = (
        resumen["prop_cancelados"] * 100
    ).round(1)

    print(f"\n{var}")
    display(resumen.head(50))
    


### Variables que podriamos eliminar del modelo de prediccion
* hotel: no le veo importancia las cancelaciones son iguales para las 2 categorias de la variable hotel


### Variables que parecen tener importancia en la prediccion de cancelaciones
* arrival_date_month: se ve que los meses de enero, febrero y marzo tienen una mayor proporción de cancelaciones.
* meal: las cancelaciones son mayores para los clientes que no tienen meal plan, y menores para los que tienen meal plan completo (BB, FB, HB). Creo que seria adecuando convertir esta variable en dicotomica (tiene meal o no tiene)
* customer_type: recategorizar en 2 categorias: "transient" y "no transient" porque hay un comportamiento diferente segun la categoria

### Revisar si es conveniente una recategorizacion
* market_segment: revisar si es conveniente agrupar los segmentos de mercado en 2 categorias: online y offline. porque hay un corpontamiento diferente segun la categoria
* reserved_room_type: revisar si es conveniente agrupar los tipos de habitacion en 2 categorias: standard y no standard. porque hay un corpontamiento diferente segun la categoria
* deposit_type: mirar... esta todo en muy concentrado en No deposit.
* agento y company: estan muy atomizadas. recategorizamos en si/no?


## Variables numericas
No usaremos:
* 'booking_changes' es posterior a 'booking_date'
* days_in_waiting_list es posterior a 'booking_date'

Numéricas (18): ['is_canceled', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


In [ ]:
var_num = ('lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests')

n = len(var_num)
ncols = 3
nrows = math.ceil(n / ncols)

df["cancelacion"] = df["is_canceled"].map(
    {0: "No cancelado", 1: "Cancelado"})

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(5*ncols, 4*nrows)
)

axes = axes.flatten()

for i, var in enumerate(var_num):

    sns.boxplot(
        data=df,
        x="cancelacion",
        y=var,
        ax=axes[i],
        showfliers = False   # esta opcion quita los outliers del box plot, para que se vea mejor la distribucion de los datos
        ) 

    axes[i].set_title(var)
    axes[i].set_xlabel("")
    
# Ocultar paneles vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
var_num_pocosVal = ('adults', 'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'required_car_parking_spaces')

for var in var_num_pocosVal:

    resumen = (
    df
    .groupby(var, dropna=False)
    .agg(
        n=("is_canceled", "size"),
        cancelaciones=("is_canceled", "sum"),
        prop_cancelados=("is_canceled", "mean")
    )
)

    resumen["prop_cancelados"] = (
        resumen["prop_cancelados"] * 100
    ).round(1)

    print(f"\n{var}")
    display(resumen.head(10))

In [ ]:
df["has_children_or_babies"] = (
    (df["children"] > 0) | (df["babies"] > 0)
).astype(int)

df.groupby("has_children_or_babies").agg(
    n=("is_canceled", "size"),
    cancelaciones=("is_canceled", "sum"),
    prop_cancelacion=("is_canceled", "mean")
)

In [ ]:
df["has_previous_bookings"] = (
    (df["previous_bookings_not_canceled"] > 0) | (df["previous_cancellations"] > 0)
).astype(int)

df.groupby("has_previous_bookings").agg(
    n=("is_canceled", "size"),
    cancelaciones=("is_canceled", "sum"),
    prop_cancelacion=("is_canceled", "mean")
)

In [ ]:
df["has_previous_bookings_not_canceled"] = (
    (df["previous_bookings_not_canceled"] > 0)
).astype(int)

df.groupby("has_previous_bookings_not_canceled").agg(
    n=("is_canceled", "size"),
    cancelaciones=("is_canceled", "sum"),
    prop_cancelacion=("is_canceled", "mean")
)

### Variables que podriamos quitar del modelo

adults
children
babies
is_repeated_guest
previous_cancellation
previous_booking_not_cancelled
required_car_parking_spaces


In [ ]:
market_cancel = (
    df.groupby("market_segment")
      .agg(
          n=("is_canceled", "size"),
          prop_cancelados=("is_canceled", "mean")
      )
      .reset_index()
)

market_cancel = (
    market_cancel[market_cancel["n"] > 10]
    .sort_values("prop_cancelados", ascending=False)
)

plt.figure(figsize=(10, 5))
bars = plt.bar(
    market_cancel["market_segment"],
    market_cancel["prop_cancelados"]
)

for _, row in market_cancel.iterrows():
    x = market_cancel.index.get_loc(_)

# Agregar porcentaje arriba de la barra
for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y,
        f"{y:.1%}",
        ha="center",
        va="bottom"
    )

# Agregar n en la base de la barra
for bar, n in zip(bars, market_cancel["n"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        0.01,                # altura fija cerca de la base
        f"n={n}",
        ha="center",
        va="bottom",
        fontsize=8,
        color="black"
    )

#plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.title("Proporción de cancelados por Market Segment")
plt.xlabel("Market Segment")
plt.ylabel("% de cancelados")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
room_cancel = (
    df.groupby("reserved_room_type")
      .agg(
          n=("is_canceled", "size"),
          prop_cancelados=("is_canceled", "mean")
      )
      .reset_index()
)

# Mantener categorías con más de 10 observaciones
room_cancel = room_cancel[room_cancel["n"] > 10]

# Orden alfabético
room_cancel = room_cancel.sort_values("reserved_room_type")

plt.figure(figsize=(10, 5))
bars = plt.bar(
    room_cancel["reserved_room_type"],
    room_cancel["prop_cancelados"]
)

# Porcentaje arriba de la barra
for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        y,
        f"{y:.1%}",
        ha="center",
        va="bottom"
    )

# n en la base de la barra
for bar, n in zip(bars, room_cancel["n"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        0.01,
        f"n={n}",
        ha="center",
        va="bottom",
        fontsize=8
    )

#plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.title("Proporción de cancelados por Reserved Room Type")
plt.xlabel("Reserved Room Type")
plt.ylabel("% de cancelados")
plt.tight_layout()
plt.show()

In [ ]:
df["len_stays"] = (
    df["stays_in_week_nights"] + df["stays_in_weekend_nights"]
).astype(int)

df["len_stays_cat"] = pd.cut(
    df["len_stays"],
    bins=7
)

df["len_stays_cat"].value_counts().sort_index()

resumen = (
    df.groupby("len_stays_cat", observed=False)
      .agg(
          n=("is_canceled", "size"),
          cancelaciones=("is_canceled", "sum"),
          prop_cancelacion=("is_canceled", "mean")
      )
)

resumen


In [ ]:
# df["len_stays_cat2"] = pd.cut(
#     df["len_stays"][df["len_stays"] < 10],
#     bins=5
# )

df["len_stays"].value_counts().sort_index()

resumen = (
    df.groupby("len_stays", observed=False)
      .agg(
          n=("is_canceled", "size"),
          cancelaciones=("is_canceled", "sum"),
          prop_cancelacion=("is_canceled", "mean")
      )
)

resumen

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    resumen.index.astype(str),
    resumen["prop_cancelacion"]
)

plt.xlabel("Duración de la estadía")
plt.ylabel("Proporción de cancelación")
plt.title("Proporción de cancelación según duración de la estadía")
plt.xticks(rotation=45)

plt.show()

## *lead_time* podria incluirse pero categorizada
lead_time dentro de los 10 dias tiene menor taza de cancelacion
esto ya podia verse en los boxplot... no hay un corte muy definido pero creo que vale la pena considerarlo 

In [ ]:
df["booking_day_arrive"] = (df["lead_time"] <= 10).astype(int)


resumen = (
    df.groupby("booking_day_arrive", observed=False)
      .agg(
          n=("is_canceled", "size"),
          cancelaciones=("is_canceled", "sum"),
          prop_cancelacion=("is_canceled", "mean")
      )
)

resumen




## Correlacion entre variables numericas + "is_canceled"

In [ ]:
# Correlación entre variables numéricas
num_cols = [
    "is_canceled",
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "len_stays"
]

df[num_cols].corr().round(2)

# recategorizacion de la variable country

In [ ]:
# Portugal vs resto
df["country_grp"] = np.where(
    df["country"] == "PRT",   # o "Portugal" según cómo esté codificado
    "Portugal",
    "Resto de países"
)

# Tabla de distribución
tabla = (
    df.groupby("country_grp")
      .agg(
          n=("country_grp", "size"),
          cancelados=("is_canceled", "sum"),
          prop_cancelados=("is_canceled", "mean")
      )
      .reset_index()
)

# % sobre el total
tabla["pct_total"] = 100 * tabla["n"] / tabla["n"].sum()

# % cancelados
tabla["prop_cancelados"] = 100 * tabla["prop_cancelados"]

# Ordenar
tabla = tabla.sort_values("n", ascending=False)

print(tabla)